In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
print("sklearn:", sklearn.__version__)

sns.set_style("whitegrid")
%matplotlib inline

sklearn: 1.7.2


In [2]:
df = pd.read_csv("../data/AB_NYC_2019.csv")
print("Shape:", df.shape)
df.head()

Shape: (48895, 16)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nPrice stats:")
print(df["price"].describe())

print("\nUnique counts:")
for col in ["neighbourhood_group", "neighbourhood", "room_type"]:
    print(f"  {col}: {df[col].nunique()}")

Missing values:
name                    16
host_name               21
last_review          10052
reviews_per_month    10052
dtype: int64

Price stats:
count    48895.000000
mean       152.720687
std        240.154170
min          0.000000
25%         69.000000
50%        106.000000
75%        175.000000
max      10000.000000
Name: price, dtype: float64

Unique counts:
  neighbourhood_group: 5
  neighbourhood: 221
  room_type: 3


In [4]:
df_clean = df.copy()

# Drop identifier + free-text columns
df_clean = df_clean.drop(columns=["id", "host_id", "host_name", "name", "last_review"])

# Drop rows with price == 0
df_clean = df_clean[df_clean["price"] > 0].reset_index(drop=True)

# Fill missing reviews_per_month with 0 (means "no reviews yet")
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(0)

# Clip outliers at 99th percentile
price_cap = df_clean["price"].quantile(0.99)
nights_cap = df_clean["minimum_nights"].quantile(0.99)
host_cap = df_clean["calculated_host_listings_count"].quantile(0.99)

df_clean["price"] = df_clean["price"].clip(upper=price_cap)
df_clean["minimum_nights"] = df_clean["minimum_nights"].clip(upper=nights_cap)
df_clean["calculated_host_listings_count"] = df_clean["calculated_host_listings_count"].clip(upper=host_cap)

# Create log-transformed target
df_clean["price_log"] = np.log1p(df_clean["price"])

print("Shape after cleaning:", df_clean.shape)
print("Missing values:", df_clean.isnull().sum().sum())
print("\nClipping caps — price: {}, nights: {}, host_count: {}".format(
    round(price_cap), round(nights_cap), round(host_cap)
))

Shape after cleaning: (48884, 12)
Missing values: 0

Clipping caps — price: 799, nights: 45, host_count: 232


In [5]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "neighbourhood_group", "neighbourhood", "latitude", "longitude",
    "room_type", "minimum_nights", "number_of_reviews",
    "reviews_per_month", "calculated_host_listings_count",
    "availability_365"
]

X = df_clean[feature_cols]
y = df_clean["price_log"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Column groups
num_cols = ["latitude", "longitude", "minimum_nights", "number_of_reviews",
            "reviews_per_month", "calculated_host_listings_count", "availability_365"]
cat_cols = ["neighbourhood_group", "room_type"]
high_card_col = ["neighbourhood"]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (39107, 10)
Test size: (9777, 10)


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from category_encoders import TargetEncoder

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

high_card_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("target_enc", TargetEncoder(smoothing=10.0))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
    ("high", high_card_pipe, high_card_col)
])

print("Preprocessor built successfully")

Preprocessor built successfully


In [7]:
import joblib
import gzip
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        max_depth=25,
        max_features="log2",
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ))
])

print("🚀 Training Random Forest...")
t0 = time.time()
pipe.fit(X_train, y_train)
print(f"   ✅ Trained in {time.time()-t0:.1f}s")

# Evaluate
y_test_pred_log = pipe.predict(X_test)
y_test_pred = np.expm1(y_test_pred_log)
y_test_real = np.expm1(y_test)

print(f"\nTest R² (log-space): {r2_score(y_test, y_test_pred_log):.4f}")
print(f"Test MAE ($): {mean_absolute_error(y_test_real, y_test_pred):.2f}")
print(f"Test RMSE ($): {np.sqrt(mean_squared_error(y_test_real, y_test_pred)):.2f}")

🚀 Training Random Forest...
   ✅ Trained in 1.0s

Test R² (log-space): 0.6246
Test MAE ($): 47.56
Test RMSE ($): 93.37


In [8]:
pipe_small = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=20,
        max_features="log2",
        min_samples_split=5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    ))
])

print("🚀 Training lighter Random Forest...")
t0 = time.time()
pipe_small.fit(X_train, y_train)
print(f"   ✅ Trained in {time.time()-t0:.1f}s")

# Evaluate
y_pred_log = pipe_small.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_real = np.expm1(y_test)

print(f"\nTest R² (log-space): {r2_score(y_test, y_pred_log):.4f}")
print(f"Test MAE ($): {mean_absolute_error(y_real, y_pred):.2f}")
print(f"Test RMSE ($): {np.sqrt(mean_squared_error(y_real, y_pred)):.2f}")

🚀 Training lighter Random Forest...
   ✅ Trained in 0.5s

Test R² (log-space): 0.6240
Test MAE ($): 47.63
Test RMSE ($): 93.55


In [9]:
import os

# Remove old model file if it exists
old_path = "../models/airbnb_price_pipeline.pkl.gz"
if os.path.exists(old_path):
    os.remove(old_path)

# Save the new one
with gzip.open(old_path, "wb", compresslevel=3) as f:
    joblib.dump(pipe_small, f)

size_mb = os.path.getsize(old_path) / 1024 / 1024
print(f"✅ Model saved to: {old_path}")
print(f"File size: {size_mb:.2f} MB")

# Sanity check — reload and predict first 5 test samples
with gzip.open(old_path, "rb") as f:
    loaded = joblib.load(f)

sample_pred = np.expm1(loaded.predict(X_test.iloc[:5]))
sample_real = np.expm1(y_test.iloc[:5]).values

print("\nPredictions ($):", sample_pred.round(2))
print("Actual ($):     ", sample_real.round(2))

✅ Model saved to: ../models/airbnb_price_pipeline.pkl.gz
File size: 28.35 MB

Predictions ($): [99.06 88.92 97.41 57.45 94.62]
Actual ($):      [99. 90. 80. 60. 90.]
